# Notebook 03 v2 — CIC-IDS2017 calibration (hybrid Platt/isotonic)

**What this notebook does:**

Fits per-class calibrators on the calibration set carved out in Notebook 01-CIC v2 (M6 fix), applies them to the test set, and reports per-class calibration quality.

**Calibration strategy (hybrid Platt/isotonic, decided in response to TA's rare-class caveat):**

- **Isotonic regression** for classes with n_calib ≥ 30
- **Platt scaling** for classes with n_calib < 30

CIC calibration set sizes:
- Normal: 32,152 → isotonic
- DoS: 5,376 → isotonic
- Probe: 2,248 → isotonic
- R2L: 223 → isotonic
- **U2R: 7 → Platt** (the rare class needing Platt, smaller than NSL's n=10)

**Why CIC U2R is the most stringent test:** n=7 is the smallest calibration class across all three datasets. Platt's 2-parameter fit on n=7 is at the lower bound of statistical feasibility. We report it honestly.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASET = 'cic_ids2017_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PROBS_DIR = MODELS_DIR / 'probabilities'   # CIC splits predictions and probabilities like UNSW
CALIB_OUT_DIR = Path(REPO) / 'calibrators' / DATASET
CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)

PLATT_THRESHOLD = 30
ECE_N_BINS = 10

print(f'Dataset: {DATASET}')
print(f'Probabilities source: {PROBS_DIR}')
print(f'Output dir: {CALIB_OUT_DIR}')

Dataset: cic_ids2017_v2
Probabilities source: /content/drive/MyDrive/XIDS_Research/xids-research/models/cic_ids2017_v2/probabilities
Output dir: /content/drive/MyDrive/XIDS_Research/xids-research/calibrators/cic_ids2017_v2


## 2. Load labels (with defensive schema handling)

In [3]:
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)

print(f'class_mappings.json keys: {list(class_info.keys())}')

# CIC may use different key names than NSL/UNSW
mapping_5 = None
for key_candidate in ['multiclass_5', 'five_class_id_to_name', '5class', 'five_class', 'multiclass',
                       'cic_id_to_name', 'class_id_to_name', 'id_to_name']:
    if key_candidate in class_info:
        mapping_5 = class_info[key_candidate]
        print(f"Using key: '{key_candidate}'")
        break

if mapping_5 is None:
    print(f'\nFull JSON content:')
    print(json.dumps(class_info, indent=2))
    raise KeyError('No 5-class mapping found. Update the candidate list.')

INT_TO_CATEGORY = {int(k): v for k, v in mapping_5.items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'\nCalibration set: {len(y_calib_b):,}, Test set: {len(y_test_b):,}')
print(f'5-class names: {CLASS_NAMES_5}')
print()

# Per-class sizes
calib_counts_5 = Counter(y_calib_5)
calib_counts_b = Counter(y_calib_b)

print('Per-class calibration set sizes:')
print(f'  Binary:')
for c in [0, 1]:
    n = calib_counts_b[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {"Normal" if c == 0 else "Attack":8s}: n={n:>6,}  → {strat}')

print(f'  5-class:')
for c in range(5):
    n = calib_counts_5[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {CLASS_NAMES_5[c]:8s}: n={n:>6,}  → {strat}')

# Class prior comparison for context
test_counts_5 = Counter(y_test_5)
n_calib = len(y_calib_5)
n_test = len(y_test_5)
print()
print('CIC class-prior comparison (for distribution-shift context):')
print(f'  {"Class":<8} {"p_calib (%)":>12} {"p_test (%)":>12} {"ratio":>8}')
for c in range(5):
    p_calib = calib_counts_5[c] / n_calib
    p_test = test_counts_5[c] / n_test
    ratio = p_test / p_calib if p_calib > 0 else float('inf')
    print(f'  {CLASS_NAMES_5[c]:<8} {100*p_calib:>12.2f} {100*p_test:>12.2f} {ratio:>8.2f}x')

class_mappings.json keys: ['binary', 'multiclass_5', 'label_mapping', 'note']
Using key: 'multiclass_5'

Calibration set: 40,006, Test set: 40,007
5-class names: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

Per-class calibration set sizes:
  Binary:
    Normal  : n=32,152  → isotonic
    Attack  : n= 7,854  → isotonic
  5-class:
    Normal  : n=32,152  → isotonic
    DoS     : n= 5,376  → isotonic
    Probe   : n= 2,248  → isotonic
    R2L     : n=   223  → isotonic
    U2R     : n=     7  → Platt

CIC class-prior comparison (for distribution-shift context):
  Class     p_calib (%)   p_test (%)    ratio
  Normal          80.37        80.37     1.00x
  DoS             13.44        13.44     1.00x
  Probe            5.62         5.62     1.00x
  R2L              0.56         0.56     1.00x
  U2R              0.02         0.02     1.00x


## 3. Helper functions (identical to NSL hybrid notebook)

In [4]:
def fit_calibrator(p_calib, y_indicator, n_class):
    if n_class >= PLATT_THRESHOLD:
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    else:
        cal = LogisticRegression(C=1e10, solver='lbfgs')
        cal.fit(p_calib.reshape(-1, 1), y_indicator)
        return cal, 'platt'

def apply_calibrator(calibrator, strategy, p_test):
    if strategy == 'isotonic':
        return calibrator.predict(p_test)
    else:
        return calibrator.predict_proba(p_test.reshape(-1, 1))[:, 1]

def expected_calibration_error(probs, labels, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (probs >= lo) & (probs <= hi) if i == n_bins - 1 else (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(probs[mask].mean() - labels[mask].mean())
    return float(ece)

def calibrate_model_per_class(p_calib_2d, y_calib, p_test_2d, y_test, n_classes):
    p_test_cal = np.zeros_like(p_test_2d)
    strategies, brier_pre, brier_post, ece_pre, ece_post = {}, {}, {}, {}, {}
    calib_counts = Counter(y_calib)

    for c in range(n_classes):
        y_calib_c = (y_calib == c).astype(int)
        y_test_c = (y_test == c).astype(int)
        n_c = calib_counts[c]
        cal, strat = fit_calibrator(p_calib_2d[:, c], y_calib_c, n_c)
        p_test_cal[:, c] = apply_calibrator(cal, strat, p_test_2d[:, c])
        strategies[c] = strat
        brier_pre[c] = float(brier_score_loss(y_test_c, p_test_2d[:, c]))
        brier_post[c] = float(brier_score_loss(y_test_c, p_test_cal[:, c]))
        ece_pre[c] = expected_calibration_error(p_test_2d[:, c], y_test_c, ECE_N_BINS)
        ece_post[c] = expected_calibration_error(p_test_cal[:, c], y_test_c, ECE_N_BINS)

    row_sums = p_test_cal.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    p_test_cal_renorm = p_test_cal / row_sums

    return {
        'p_test_cal': p_test_cal_renorm,
        'strategies': strategies,
        'brier_pre': brier_pre, 'brier_post': brier_post,
        'ece_pre': ece_pre, 'ece_post': ece_post,
    }

def calibrate_binary(p_calib_2d, y_calib, p_test_2d, y_test):
    p_calib_attack = p_calib_2d[:, 1]
    p_test_attack = p_test_2d[:, 1]
    calib_counts = Counter(y_calib)
    n_attack = calib_counts[1]
    cal, strat = fit_calibrator(p_calib_attack, y_calib, n_attack)
    p_test_cal_attack = apply_calibrator(cal, strat, p_test_attack)
    p_test_cal_attack = np.clip(p_test_cal_attack, 0, 1)
    p_test_cal_2d = np.column_stack([1 - p_test_cal_attack, p_test_cal_attack])
    return {
        'p_test_cal': p_test_cal_2d,
        'strategies': {1: strat},
        'brier_pre': {1: float(brier_score_loss(y_test, p_test_attack))},
        'brier_post': {1: float(brier_score_loss(y_test, p_test_cal_attack))},
        'ece_pre': {1: expected_calibration_error(p_test_attack, y_test, ECE_N_BINS)},
        'ece_post': {1: expected_calibration_error(p_test_cal_attack, y_test, ECE_N_BINS)},
    }

print('✓ Helper functions loaded')

✓ Helper functions loaded


## 4. Calibrate all 9 models

In [5]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task})...')
    p_calib = np.load(PROBS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PROBS_DIR / f'{name}_test_proba.npy')

    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])

    if task == 'binary':
        result = calibrate_binary(p_calib, y_calib_b, p_test, y_test_b)
    else:
        result = calibrate_model_per_class(p_calib, y_calib_5, p_test, y_test_5, n_classes=5)

    np.save(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy', result['p_test_cal'])

    print(f'  Strategies: {result["strategies"]}')
    if task == '5class':
        for c in range(5):
            print(f'  {CLASS_NAMES_5[c]:8s}: Brier {result["brier_pre"][c]:.4f} → {result["brier_post"][c]:.4f}'
                  f'   ECE {result["ece_pre"][c]:.4f} → {result["ece_post"][c]:.4f}'
                  f'   [{result["strategies"][c]}]')
    else:
        print(f'  Attack  : Brier {result["brier_pre"][1]:.4f} → {result["brier_post"][1]:.4f}'
              f'   ECE {result["ece_pre"][1]:.4f} → {result["ece_post"][1]:.4f}'
              f'   [{result["strategies"][1]}]')

    calibration_summary[name] = {
        'task': task,
        'strategies': {int(k): v for k, v in result['strategies'].items()},
        'brier_pre':  {int(k): v for k, v in result['brier_pre'].items()},
        'brier_post': {int(k): v for k, v in result['brier_post'].items()},
        'ece_pre':  {int(k): v for k, v in result['ece_pre'].items()},
        'ece_post': {int(k): v for k, v in result['ece_post'].items()},
    }

with open(CALIB_OUT_DIR / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ All 9 models calibrated. Summary: {CALIB_OUT_DIR}/calibration_summary.json')


Calibrating rf_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.0014 → 0.0013   ECE 0.0016 → 0.0005   [isotonic]

Calibrating xgb_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.0009 → 0.0009   ECE 0.0007 → 0.0001   [isotonic]

Calibrating dnn_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.0208 → 0.0159   ECE 0.0198 → 0.0017   [isotonic]

Calibrating rf_5class_smote (5class)...
  Strategies: {0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}
  Normal  : Brier 0.0016 → 0.0014   ECE 0.0029 → 0.0004   [isotonic]
  DoS     : Brier 0.0008 → 0.0006   ECE 0.0015 → 0.0003   [isotonic]
  Probe   : Brier 0.0004 → 0.0003   ECE 0.0005 → 0.0001   [isotonic]
  R2L     : Brier 0.0004 → 0.0004   ECE 0.0012 → 0.0002   [isotonic]
  U2R     : Brier 0.0000 → 0.0000   ECE 0.0001 → 0.0001   [platt]

Calibrating xgb_5class_smote (5class)...
  Strategies: {0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 

## 5. Summary table

In [6]:
rows = []
for name, task in ALL_MODELS:
    s = calibration_summary[name]
    if task == '5class':
        brier_pre_macro = np.mean(list(s['brier_pre'].values()))
        brier_post_macro = np.mean(list(s['brier_post'].values()))
        ece_pre_macro = np.mean(list(s['ece_pre'].values()))
        ece_post_macro = np.mean(list(s['ece_post'].values()))
        platt_classes = [CLASS_NAMES_5[c] for c, st in s['strategies'].items() if st == 'platt']
        rows.append({
            'Model': name,
            'Task': '5-class',
            'Brier pre (macro)':  round(brier_pre_macro,  5),
            'Brier post (macro)': round(brier_post_macro, 5),
            'ECE pre (macro)':    round(ece_pre_macro,    5),
            'ECE post (macro)':   round(ece_post_macro,   5),
            'Platt for': ','.join(platt_classes) if platt_classes else '—',
        })
    else:
        rows.append({
            'Model': name,
            'Task': 'binary',
            'Brier pre (macro)':  round(s['brier_pre'][1],  5),
            'Brier post (macro)': round(s['brier_post'][1], 5),
            'ECE pre (macro)':    round(s['ece_pre'][1],    5),
            'ECE post (macro)':   round(s['ece_post'][1],   5),
            'Platt for': '—' if s['strategies'][1] == 'isotonic' else 'attack class',
        })

df = pd.DataFrame(rows)
print('\nCIC v2 — Hybrid Platt/Isotonic Calibration Results')
print('=' * 110)
print(df.to_string(index=False))
print('=' * 110)

TABLES_DIR = Path(REPO) / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
csv_path = TABLES_DIR / 'cic_v2_calibration_summary.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')


CIC v2 — Hybrid Platt/Isotonic Calibration Results
           Model    Task  Brier pre (macro)  Brier post (macro)  ECE pre (macro)  ECE post (macro) Platt for
    rf_binary_cw  binary            0.00141             0.00131          0.00159           0.00047         —
   xgb_binary_cw  binary            0.00090             0.00090          0.00072           0.00012         —
   dnn_binary_cw  binary            0.02081             0.01594          0.01985           0.00171         —
 rf_5class_smote 5-class            0.00063             0.00056          0.00125           0.00024       U2R
xgb_5class_smote 5-class            0.00044             0.00041          0.00043           0.00016       U2R
dnn_5class_smote 5-class            0.01124             0.00538          0.01304           0.00081       U2R
    rf_5class_cw 5-class            0.00060             0.00060          0.00078           0.00029       U2R
   xgb_5class_cw 5-class            0.00036             0.00035          0.0

## 6. Per-class breakdown (5-class models only)

In [7]:
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    s = calibration_summary[name]
    for c in range(5):
        perclass_rows.append({
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts_5[c],
            'Strategy': s['strategies'][c],
            'Brier pre':  round(s['brier_pre'][c],  5),
            'Brier post': round(s['brier_post'][c], 5),
            'ECE pre':    round(s['ece_pre'][c],    5),
            'ECE post':   round(s['ece_post'][c],   5),
        })

df_perclass = pd.DataFrame(perclass_rows)
print('\nCIC v2 — Per-class calibration breakdown')
print('=' * 120)
print(df_perclass.to_string(index=False))
print('=' * 120)

csv_path = TABLES_DIR / 'cic_v2_calibration_perclass.csv'
df_perclass.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')


CIC v2 — Per-class calibration breakdown
           Model  Class  n_calib Strategy  Brier pre  Brier post  ECE pre  ECE post
 rf_5class_smote Normal    32152 isotonic    0.00155     0.00138  0.00294   0.00044
 rf_5class_smote    DoS     5376 isotonic    0.00078     0.00062  0.00152   0.00031
 rf_5class_smote  Probe     2248 isotonic    0.00038     0.00034  0.00053   0.00014
 rf_5class_smote    R2L      223 isotonic    0.00040     0.00038  0.00120   0.00024
 rf_5class_smote    U2R        7    platt    0.00003     0.00005  0.00008   0.00005
xgb_5class_smote Normal    32152 isotonic    0.00108     0.00101  0.00104   0.00016
xgb_5class_smote    DoS     5376 isotonic    0.00050     0.00043  0.00050   0.00025
xgb_5class_smote  Probe     2248 isotonic    0.00030     0.00030  0.00025   0.00019
xgb_5class_smote    R2L      223 isotonic    0.00030     0.00027  0.00031   0.00015
xgb_5class_smote    U2R        7    platt    0.00003     0.00003  0.00003   0.00002
dnn_5class_smote Normal    32152 i

## 7. Sanity checks

In [8]:
print('Sanity checks on calibrated probabilities:')
print('-' * 70)
for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy')
    expected_cols = 2 if task == 'binary' else 5
    shape_ok = p_cal.shape[1] == expected_cols
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())
    finite_ok = np.isfinite(p_cal).all()
    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

Sanity checks on calibrated probabilities:
----------------------------------------------------------------------
  ✓ rf_binary_cw           shape=(40007, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_binary_cw          shape=(40007, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_binary_cw          shape=(40007, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_smote        shape=(40007, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_smote       shape=(40007, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_smote       shape=(40007, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_cw           shape=(40007, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_cw          shape=(40007, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_cw          shape=(40007, 5)  range_ok=True  sum_max_dev=0.00000  finite=True


## 8. Commit

In [9]:
os.chdir(REPO)
!git add notebooks/03_cic_calibration_v2.ipynb
!git add calibrators/cic_ids2017_v2/
!git add results/tables/cic_v2_calibration_summary.csv
!git add results/tables/cic_v2_calibration_perclass.csv
!git status --short
!git commit -m 'Notebook 03-CIC v2: hybrid Platt/isotonic calibration (U2R n=7 → Platt)'
!git push origin main

Refresh index: 100% (251/251), done.
A  calibrators/cic_ids2017_v2/calibration_summary.json
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
 M notebooks/02_cic_train_models_v2.ipynb
 M notebooks/02_nsl_rf_retrain_v2.ipynb
 M notebooks/02_train_models_v2.ipynb
 M notebooks/02_unsw_train_models_v2.ipynb
AM notebooks/03_cic_calibration_v2.ipynb
 M notebooks/03_nsl_calibration_v2.ipynb
 M notebooks/03c_nsl_calibration_dirichlet_v2.ipynb
 M notebooks/03c_unsw_calibration_dirichlet_v2.ipynb
A  results/tables/cic_v2_calibration_perclass.csv
A  results/tables/cic_v2_calibration_summary.csv
?? calibrators/nsl_kdd/
?? calibrators/unsw_nb15/
?? docs/v2_rebuild_progress_day1_day2_v7.md
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? notebooks/03b_nsl_calibration_diagnostic.ipynb
?? results/figures/diag_nsl_dnn5smote_normal_reliability.png
?? results/figures/diag_nsl_rf5smote_normal_reliability.png
?? re

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Verify it worked
import os
path = '/content/drive/MyDrive/XIDS_Research/xids-research/data/processed/cic_ids2017_v2'
print(f'Path exists: {os.path.exists(path)}')
if os.path.exists(path):
    files = os.listdir(path)
    print(f'Files: {len(files)}')
    print('y_test_5class.npy' in files)

Mounted at /content/drive
Path exists: True
Files: 16
True


In [4]:
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
PROBS_DIR = Path(REPO) / 'models' / 'cic_ids2017_v2' / 'probabilities'
HYBRID_DIR = Path(REPO) / 'calibrators' / 'cic_ids2017_v2'  # hybrid, not dirichlet
PROCESSED = Path(REPO) / 'data' / 'processed' / 'cic_ids2017_v2'

y_test = np.load(PROCESSED / 'y_test_5class.npy')

models_5class = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
                 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']

print('CIC HYBRID Platt/Isotonic — argmax preservation check')
print(f'{"Model":<22} {"% flipped":>10} {"Acc pre":>9} {"Acc post":>10} {"Δ acc":>9} {"F1m pre":>9} {"F1m post":>10} {"Δ F1m":>9}')
print('-' * 95)
for name in models_5class:
    p_pre = np.load(PROBS_DIR / f'{name}_test_proba.npy')
    p_post = np.load(HYBRID_DIR / f'{name}_test_proba_calibrated.npy')
    pred_pre = p_pre.argmax(axis=1)
    pred_post = p_post.argmax(axis=1)
    n_flipped = (pred_pre != pred_post).sum()
    pct_flipped = 100 * n_flipped / len(pred_pre)

    acc_pre = accuracy_score(y_test, pred_pre)
    acc_post = accuracy_score(y_test, pred_post)
    f1_pre = f1_score(y_test, pred_pre, average='macro', zero_division=0)
    f1_post = f1_score(y_test, pred_post, average='macro', zero_division=0)

    print(f'{name:<22} {pct_flipped:>9.2f}% {acc_pre:>9.4f} {acc_post:>10.4f} {acc_post-acc_pre:>+9.4f} '
          f'{f1_pre:>9.4f} {f1_post:>10.4f} {f1_post-f1_pre:>+9.4f}')

CIC HYBRID Platt/Isotonic — argmax preservation check
Model                   % flipped   Acc pre   Acc post     Δ acc   F1m pre   F1m post     Δ F1m
-----------------------------------------------------------------------------------------------
rf_5class_smote             0.02%    0.9983     0.9984   +0.0001    0.9571     0.9573   +0.0003
xgb_5class_smote            0.01%    0.9989     0.9989   +0.0000    0.9782     0.9778   -0.0004
dnn_5class_smote            3.44%    0.9606     0.9802   +0.0197    0.7265     0.8774   +0.1509
rf_5class_cw                0.05%    0.9982     0.9983   +0.0001    0.9584     0.7917   -0.1667
xgb_5class_cw               0.01%    0.9990     0.9991   +0.0001    0.9799     0.9817   +0.0018
dnn_5class_cw              19.34%    0.8227     0.9461   +0.1235    0.5400     0.6830   +0.1430


In [5]:
import os
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
HYBRID_DIR = Path(REPO) / 'calibrators' / 'cic_ids2017_v2'
DIRICHLET_DIR = Path(REPO) / 'calibrators' / 'cic_ids2017_v2_dirichlet'

print(f'Hybrid dir ({HYBRID_DIR}):')
print(f'  Exists: {HYBRID_DIR.exists()}')
if HYBRID_DIR.exists():
    files = sorted(os.listdir(HYBRID_DIR))
    print(f'  {len(files)} files:')
    for f in files:
        size = (HYBRID_DIR / f).stat().st_size
        print(f'    {f:<45} {size:>10,} bytes')

print()
print(f'Dirichlet dir ({DIRICHLET_DIR}):')
print(f'  Exists: {DIRICHLET_DIR.exists()}')
if DIRICHLET_DIR.exists():
    files = sorted(os.listdir(DIRICHLET_DIR))
    print(f'  {len(files)} files:')
    for f in files:
        size = (DIRICHLET_DIR / f).stat().st_size
        print(f'    {f:<45} {size:>10,} bytes')

# Now compare actual file contents on one model
import numpy as np
name = 'dnn_5class_cw'
hybrid_f = HYBRID_DIR / f'{name}_test_proba_calibrated.npy'
dirichlet_f = DIRICHLET_DIR / f'{name}_test_proba_calibrated.npy'

if hybrid_f.exists() and dirichlet_f.exists():
    p_h = np.load(hybrid_f)
    p_d = np.load(dirichlet_f)
    print(f'\n{name} comparison:')
    print(f'  Hybrid shape: {p_h.shape}, first row: {p_h[0]}')
    print(f'  Dirichlet shape: {p_d.shape}, first row: {p_d[0]}')
    print(f'  Max abs diff: {np.abs(p_h - p_d).max()}')
    print(f'  Files identical: {np.allclose(p_h, p_d)}')

Hybrid dir (/content/drive/MyDrive/XIDS_Research/xids-research/calibrators/cic_ids2017_v2):
  Exists: True
  10 files:
    calibration_summary.json                           6,751 bytes
    dnn_5class_cw_test_proba_calibrated.npy          800,268 bytes
    dnn_5class_smote_test_proba_calibrated.npy       800,268 bytes
    dnn_binary_cw_test_proba_calibrated.npy          320,184 bytes
    rf_5class_cw_test_proba_calibrated.npy         1,600,408 bytes
    rf_5class_smote_test_proba_calibrated.npy      1,600,408 bytes
    rf_binary_cw_test_proba_calibrated.npy           640,240 bytes
    xgb_5class_cw_test_proba_calibrated.npy          800,268 bytes
    xgb_5class_smote_test_proba_calibrated.npy       800,268 bytes
    xgb_binary_cw_test_proba_calibrated.npy          320,184 bytes

Dirichlet dir (/content/drive/MyDrive/XIDS_Research/xids-research/calibrators/cic_ids2017_v2_dirichlet):
  Exists: True
  10 files:
    calibration_summary.json                           6,645 bytes
    dnn_5cl